# Validação de Dados — Tickers Problemáticos

Verifica cobertura de preços para os tickers que tiveram problema na coleta (`coleta_report.csv`).

| Parte | O que faz |
|---|---|
| 1 | Carrega composição histórica e report de coleta |
| 2 | Mapeia quando cada ticker estava no S&P 500 |
| 3 | Confere se o arquivo de preços cobre o período no índice |
| 4 | Verifica sucessores para empresas renomeadas/fundidas |
| 5 | Resumo com ações necessárias |
| 6 | Coleta Tiingo (tentativa inicial sem datas específicas) |
| 7 | Deleta arquivos do grupo [A] com ticker reciclado |
| 8 | Retry Yahoo para tickers recentes |
| 9 | **Deleta arquivos do grupo [B] com dados de período errado** |
| 10 | **Tiingo com datas específicas por ticker (principal recuperação)** |

In [46]:
import pandas as pd
from pathlib import Path

DATA_DIR      = Path("../data_bases")
PRICES_YAHOO  = DATA_DIR / "prices"
PRICES_TIINGO = DATA_DIR / "prices_tiingo"
EXTERNAL_DIR  = DATA_DIR / "external"

print("Pronto.")

Pronto.


---
## Parte 1 — Carregar dados

In [47]:
hist_sp500 = pd.read_csv(EXTERNAL_DIR / "sp500_historico.csv")
hist_sp500["date"] = pd.to_datetime(hist_sp500["date"])

# Pré-computar set de tickers por snapshot (busca mais rápida)
snap_sets = hist_sp500["tickers"].str.split(",").apply(lambda lst: {t.strip() for t in lst})

coleta = pd.read_csv(EXTERNAL_DIR / "coleta_report.csv")

print(f"Composição histórica: {len(hist_sp500)} snapshots")
print(f"  {hist_sp500['date'].min().date()} → {hist_sp500['date'].max().date()}")
print(f"\nReport de coleta: {len(coleta)} tickers")
print(coleta.groupby("status").size().rename("count").to_string())
print()
print(coleta.to_string(index=False))

Composição histórica: 2705 snapshots
  1996-01-02 → 2026-01-14

Report de coleta: 739 tickers
status
delisted    134
ok          605

ticker   status  n_rows      note
  ANTM delisted       0 sem dados
   APC delisted       0 sem dados
  ARNC delisted       0 sem dados
   BLL delisted       0 sem dados
    CA       ok     512       NaN
   CBS delisted       0 sem dados
  CDAY delisted       0 sem dados
 DISCA delisted       0 sem dados
   DNB delisted       0 sem dados
    DO delisted       0 sem dados
   EMC       ok     661       NaN
  ENDP delisted       0 sem dados
    FB       ok     131       NaN
  FBHS delisted       0 sem dados
   FRC delisted       0 sem dados
   FTR delisted       0 sem dados
   GPS delisted       0 sem dados
   HCP delisted       0 sem dados
    LB       ok     379       NaN
   MMC delisted       0 sem dados
   MNK delisted       0 sem dados
   MON delisted       0 sem dados
  PEAK delisted       0 sem dados
   PKI delisted       0 sem dados
    RE delisted 

---
## Parte 2 — Período de cada ticker no S&P 500

Usando `sp500_historico.csv`, encontramos o primeiro e último snapshot que contém cada ticker.
Se um ticker não aparece no arquivo, pode ter saído antes de 1996 (início do histórico) — improvável neste conjunto.

In [48]:
def get_period(ticker):
    """Retorna (primeira_data, última_data) de snapshot com este ticker."""
    mask  = snap_sets.apply(lambda s: ticker in s)
    dates = hist_sp500.loc[mask, "date"]
    if dates.empty:
        return pd.NaT, pd.NaT
    return dates.min(), dates.max()

rows = []
for t in sorted(coleta["ticker"]):
    first, last = get_period(t)
    st = coleta.loc[coleta["ticker"] == t, "status"].values[0]
    rows.append({
        "ticker":      t,
        "coleta":      st,
        "sp500_from":  first.strftime("%Y-%m-%d") if pd.notna(first) else "não encontrado",
        "sp500_until": last.strftime("%Y-%m-%d")  if pd.notna(last)  else "não encontrado",
    })

df_periods = pd.DataFrame(rows)
print(df_periods.to_string(index=False))

ticker   coleta sp500_from sp500_until
     A       ok 2000-06-05  2026-01-14
  AABA delisted 1999-12-08  2017-06-16
   AAL       ok 1996-01-02  2024-07-08
   AAP       ok 2015-07-09  2023-07-10
  AAPL       ok 1996-01-02  2026-01-14
  ABBV       ok 2013-01-02  2026-01-14
   ABC delisted 2001-08-30  2023-08-25
  ABMD delisted 2018-05-31  2022-12-19
  ABNB       ok 2023-09-18  2026-01-14
   ABT       ok 1996-01-02  2026-01-14
  ACGL       ok 2022-11-01  2026-01-14
   ACN       ok 2011-07-06  2026-01-14
  ADBE       ok 1997-05-06  2026-01-14
   ADI       ok 1999-10-12  2026-01-14
   ADM       ok 1996-01-02  2026-01-14
   ADP       ok 1996-01-02  2026-01-14
   ADS delisted 2013-12-23  2020-05-22
  ADSK       ok 1996-01-02  2026-01-14
   AEE       ok 1996-01-02  2026-01-14
   AEP       ok 1996-01-02  2026-01-14
   AES       ok 1998-10-02  2026-01-14
   AET       ok 1996-01-02  2018-11-28
   AFL       ok 1999-05-28  2026-01-14
   AGN delisted 1996-01-02  2020-04-06
   AIG       ok 1996-01-0

---
## Parte 3 — Cobertura do arquivo de preços vs período no índice

Para cada ticker, comparamos:
- **sp500_until**: última data em que aparece no histórico de composição
- **price_until**: última data do arquivo de preços que temos

Se `price_until < sp500_until - 15 dias` → **gap**: dados faltando para a extensão.

Tickers que saíram do índice antes de 2016-01-01 já estão cobertos pela base do professor (`Pt.csv`).

In [49]:
EXT_START = pd.Timestamp("2016-01-01")  # base do professor termina em 2015-12-30

def price_range(ticker):
    """Retorna (início, fim, fonte) do arquivo de preços, ou (NaT, NaT, None) se não existe."""
    for folder in [PRICES_YAHOO, PRICES_TIINGO]:
        f = folder / f"{ticker}.csv"
        if f.exists():
            df = pd.read_csv(f, index_col=0, parse_dates=True)
            df.index = pd.to_datetime(df.index)
            if df.index.tz is not None:
                df.index = df.index.tz_localize(None)
            return df.index.min(), df.index.max(), folder.name
    return pd.NaT, pd.NaT, None

rows_cov = []
for t in sorted(coleta["ticker"]):
    sp_first, sp_last = get_period(t)
    p_start, p_end, src = price_range(t)

    needs_ext = pd.notna(sp_last) and sp_last >= EXT_START

    if not needs_ext:
        resultado = "prof. cobre"  # saiu antes de 2016, está no Pt.csv
    elif pd.isna(p_start):
        resultado = "SEM ARQUIVO"
    else:
        lag = (sp_last - p_end).days
        resultado = "ok" if lag <= 15 else f"gap: ~{lag} dias faltando"

    rows_cov.append({
        "ticker":      t,
        "sp500_until": sp_last.strftime("%Y-%m-%d")  if pd.notna(sp_last)  else "—",
        "price_from":  p_start.strftime("%Y-%m-%d")  if pd.notna(p_start)  else "—",
        "price_until": p_end.strftime("%Y-%m-%d")    if pd.notna(p_end)    else "—",
        "fonte":       src or "—",
        "resultado":   resultado,
    })

df_cov = pd.DataFrame(rows_cov)
print(df_cov.to_string(index=False))

ticker sp500_until price_from price_until         fonte   resultado
     A  2026-01-14 2015-07-01  2025-12-31        prices          ok
  AABA  2017-06-16 2015-07-01  2019-11-06 prices_tiingo          ok
   AAL  2024-07-08 2015-07-01  2025-12-31        prices          ok
   AAP  2023-07-10 2015-07-01  2025-12-31        prices          ok
  AAPL  2026-01-14 2015-07-01  2025-12-31        prices          ok
  ABBV  2026-01-14 2015-07-01  2025-12-31        prices          ok
   ABC  2023-08-25          —           —             — SEM ARQUIVO
  ABMD  2022-12-19 2015-07-01  2023-01-03 prices_tiingo          ok
  ABNB  2026-01-14 2020-12-10  2025-12-31        prices          ok
   ABT  2026-01-14 2015-07-01  2025-12-31        prices          ok
  ACGL  2026-01-14 2015-07-01  2025-12-31        prices          ok
   ACN  2026-01-14 2015-07-01  2025-12-31        prices          ok
  ADBE  2026-01-14 2015-07-01  2025-12-31        prices          ok
   ADI  2026-01-14 2015-07-01  2025-12-31       

---
## Parte 4 — Sucessores para empresas renomeadas/fundidas

Algumas empresas do report mudaram de ticker mas continuaram no S&P 500. Verificamos:
1. O novo ticker aparece no `sp500_historico`?
2. Temos arquivo de preço para ele?

In [50]:
# Mapeamento: ticker antigo → ticker atual (empresa mesma, nome/ticker diferente)
RENAMES = {
    "ANTM":  "ELV",   # Anthem → Elevance Health (jun/2022)
    "ARNC":  "HWM",   # Arconic Inc. → Howmet Aerospace (abr/2020)
    "CBS":   "PARA",  # ViacomCBS → Paramount Global (fev/2022)
    "DISCA": "WBD",   # Discovery → Warner Bros. Discovery (abr/2022)
    "FB":    "META",  # Facebook → Meta Platforms (jun/2022)
    "HCP":   "PEAK",  # HCP → Healthpeak Properties (jan/2020)
    "PKI":   "RVTY",  # PerkinElmer → Revvity (mar/2023)
    "TMK":   "GL",    # Torchmark → Globe Life (ago/2020)
    "WRK":   "SW",    # WestRock → Smurfit WestRock (jul/2024)
}

rows_suc = []
for old, new in RENAMES.items():
    suc_first, suc_last = get_period(new)
    p_start, p_end, src = price_range(new)
    rows_suc.append({
        "antigo":       old,
        "novo":         new,
        "novo_sp500":   suc_first.strftime("%Y-%m-%d") if pd.notna(suc_first) else "não encontrado",
        "arquivo":      src if src else "NÃO TEM",
        "price_from":   p_start.strftime("%Y-%m-%d") if pd.notna(p_start) else "—",
        "price_until":  p_end.strftime("%Y-%m-%d")   if pd.notna(p_end)   else "—",
    })

df_suc = pd.DataFrame(rows_suc)
print(df_suc.to_string(index=False))

antigo novo novo_sp500       arquivo price_from price_until
  ANTM  ELV 2022-06-28        prices 2015-07-01  2025-12-31
  ARNC  HWM 2020-04-06        prices 2016-11-01  2025-12-31
   CBS PARA 2022-02-17 prices_tiingo 2015-07-01  2025-08-07
 DISCA  WBD 2022-04-11        prices 2015-07-01  2025-12-31
    FB META 2022-06-09        prices 2015-07-01  2025-12-31
   HCP PEAK 2019-11-05       NÃO TEM          —           —
   PKI RVTY 2023-05-16        prices 2015-07-01  2025-12-31
   TMK   GL 2019-08-08        prices 2015-07-01  2025-12-31
   WRK   SW 2024-07-08        prices 2015-07-01  2025-12-31


---
## Parte 5 — Resumo das ações necessárias

---
## Parte 6 — Baixar dados faltantes

Para todos os tickers da lista [A], tentamos baixar de `2015-07-01` a `2026-01-01`.  
O yfinance retorna apenas o período em que o ticker existiu (ex: APC até ago/2019, MMC até dez/2025).  
Checkpoint salvo em `coleta_report_validacao.csv` — pode rodar novamente sem repetir os já tentados.

In [51]:
import requests
import time

CHECKPOINT_VAL = EXTERNAL_DIR / "coleta_report_validacao.csv"
TIINGO_TOKEN   = "dadfd331f2cb44969b8f7468006d20ad62b13262"

# Tickers que precisam de download (lista [A] da Parte 3)
para_baixar = [r["ticker"] for r in rows_cov if r["resultado"] == "SEM ARQUIVO"]
print(f"Tickers para baixar: {len(para_baixar)}")

# Checkpoint: pular apenas os que já obtiveram status "ok"
if CHECKPOINT_VAL.exists():
    rel_val = pd.read_csv(CHECKPOINT_VAL)
    ja_ok   = set(rel_val.loc[rel_val["status"] == "ok", "ticker"])
    print(f"Checkpoint: {len(ja_ok)} já coletados com sucesso")
else:
    rel_val = pd.DataFrame(columns=["ticker", "status", "n_rows", "price_from", "price_until"])
    ja_ok   = set()

pendentes = [t for t in para_baixar if t not in ja_ok]
print(f"Pendentes: {len(pendentes)}")
print()

for ticker in pendentes:
    print(f"  {ticker}... ", end="", flush=True)
    try:
        r = requests.get(
            f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
            params={
                "startDate":    "2015-07-01",
                "endDate":      "2026-01-01",
                "token":        TIINGO_TOKEN,
                "resampleFreq": "daily",
            },
            timeout=20,
        )

        if r.status_code != 200:
            row = {"ticker": ticker, "status": f"erro_http_{r.status_code}",
                   "n_rows": 0, "price_from": "—", "price_until": "—"}
            print(f"HTTP {r.status_code}")
        else:
            data = r.json()
            if not data:
                row = {"ticker": ticker, "status": "sem dados",
                       "n_rows": 0, "price_from": "—", "price_until": "—"}
                print("sem dados")
            else:
                df = pd.DataFrame(data)
                df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
                df = df.set_index("date").sort_index()
                # adjClose do Tiingo = split-adjusted + dividend-adjusted (= Yahoo auto_adjust=True)
                close = df["adjClose"].rename(ticker).dropna()

                if len(close) > 10:
                    close.to_csv(PRICES_TIINGO / f"{ticker}.csv", header=True)
                    row = {
                        "ticker":      ticker,
                        "status":      "ok",
                        "n_rows":      len(close),
                        "price_from":  close.index[0].strftime("%Y-%m-%d"),
                        "price_until": close.index[-1].strftime("%Y-%m-%d"),
                    }
                    print(f"ok ({len(close)} linhas | {row['price_from']} → {row['price_until']})")
                else:
                    row = {"ticker": ticker, "status": "sem dados",
                           "n_rows": len(close), "price_from": "—", "price_until": "—"}
                    print(f"sem dados ({len(close)} linhas)")

    except Exception as e:
        row = {"ticker": ticker, "status": "erro",
               "n_rows": 0, "price_from": "—", "price_until": str(e)[:80]}
        print(f"erro: {e}")

    rel_val = rel_val[rel_val["ticker"] != ticker]
    rel_val = pd.concat([rel_val, pd.DataFrame([row])], ignore_index=True)
    rel_val.to_csv(CHECKPOINT_VAL, index=False)
    time.sleep(0.4)

print()
print("=== Resultado ===")
print(rel_val["status"].value_counts().to_string())
print()
print(rel_val.to_string(index=False))

Tickers para baixar: 47
Checkpoint: 79 já coletados com sucesso
Pendentes: 47

  ABC... 

sem dados
  ADS... sem dados
  ANTM... sem dados
  ANTM... sem dados
  APC... sem dados
  APC... sem dados
  BF.B... HTTP 404
  BLL... sem dados
  BLL... sem dados
  BRK.B... HTTP 404
  CBS... HTTP 404
  CBS... HTTP 404
  CDAY... HTTP 404
  CDAY... HTTP 404
  COG... sem dados
  ENDP... HTTP 404
  ENDP... HTTP 404
  FBHS... sem dados
  FBHS... sem dados
  FLT... sem dados
  FRC... HTTP 404
  FRC... HTTP 404
  GPS... sem dados
  GPS... sem dados
  HFC... HTTP 404
  MMC... sem dados
  MMC... sem dados
  MNK... HTTP 404
  MNK... HTTP 404
  PEAK... sem dados
  PEAK... sem dados
  PKI... sem dados
  PKI... sem dados
  RE... sem dados
  RE... sem dados
  SPLS... sem dados
  TMK... sem dados
  TMK... sem dados
  VIAC... ok (1116 linhas | 2015-07-01 → 2019-12-04)
  VIAC... ok (1116 linhas | 2015-07-01 → 2019-12-04)
  WLTW... sem dados
  WRK... sem dados (1 linhas)
  WRK... sem dados (1 linhas)
  WYND... ok (2642 linhas | 2015-07-01 → 2025-12-31)
  XEC... ok (1577 linhas | 2015-07-01 → 2021-10-0

---
## Parte 7 — Deletar arquivos com dados errados (ticker reciclado)

Alguns tickers coletados pelo Tiingo contêm dados da **nova empresa** que assumiu o ticker após a original sair do índice. Esses arquivos são inválidos e precisam ser removidos.

| Ticker | Problema |
|--------|----------|
| ARNC | Novo ARNC (Arconic Corp, spinoff abr/2020) ≠ velho ARNC (virou HWM) |
| DISCA | Dados pós-fusão WBD (abr/2022), não da Discovery original |
| DNB | Novo DNB re-IPO jul/2020 ≠ velho DNB (saiu do índice em 2017) |
| DO | Diamond Offshore emergiu da falência em 2022 como nova empresa |
| FTR | FTR pós-reorganização ≠ Frontier original (saiu em 2017) |
| HCP | HCP em 2021 = High Country Bancorp; a original virou PEAK em 2020 |
| MON | Ticker MON reciclado; Monsanto foi adquirida pela Bayer em 2018 |

In [52]:
# TICKER_RECICLADO = [
#     "ARNC",   # novo ARNC (Arconic Corp spinoff) ≠ velho ARNC (agora HWM)
#     "DISCA",  # dados pós-fusão WBD, não da Discovery original
#     "DNB",    # novo DNB re-IPO 2020 ≠ velho DNB que saiu em 2017
#     "DO",     # Diamond Offshore emergiu da falência como nova empresa
#     "FTR",    # FTR pós-reorganização ≠ Frontier original
#     "HCP",    # HCP em 2021 = High Country Bancorp, não Healthcare Properties
#     "MON",    # ticker reciclado; Monsanto foi adquirida pela Bayer em 2018
# ]

# rel_val = pd.read_csv(CHECKPOINT_VAL)

# print("Deletando arquivos com dados errados:")
# for ticker in TICKER_RECICLADO:
#     f = PRICES_TIINGO / f"{ticker}.csv"
#     if f.exists():
#         f.unlink()
#         print(f"  {ticker}.csv deletado")
#     else:
#         print(f"  {ticker}.csv já não existe")

#     # Marca como inválido no checkpoint
#     rel_val.loc[rel_val["ticker"] == ticker, "status"] = "deletado_ticker_reciclado"
#     rel_val.loc[rel_val["ticker"] == ticker, ["n_rows","price_from","price_until"]] = [0,"—","—"]

# rel_val.to_csv(CHECKPOINT_VAL, index=False)
# print("\nCheckpoint atualizado.")

---
## Parte 8 — Retry Yahoo para tickers recentes (saíram do índice pós-2020)

Esses tickers saíram do índice recentemente e o Yahoo Finance ainda tem o histórico deles.
O `sem dados` na Parte 6 foi provavelmente uma falha transiente — o Yahoo **não** deleta dados de tickers que ainda existiam em 2020+.

In [53]:
import yfinance as yf
import time

# Tickers que saíram do índice após 2020-01-01 e ainda estão sem arquivo
RETRY_YAHOO = [
    "ANTM",  # Anthem → ELV em jun/2022
    "BLL",   # Ball Corp, saiu do S&P 500 em abr/2022
    "CDAY",  # Ceridian, saiu em out/2023
    "FBHS",  # Fortune Brands, virou FBIN em nov/2022
    "FRC",   # First Republic Bank, falência mai/2023
    "GPS",   # The Gap, saiu em jan/2022
    "MMC",   # Marsh & McLennan — ainda no índice até dez/2025!
    "PEAK",  # Healthpeak Properties, virou DOC em fev/2024
    "PKI",   # PerkinElmer, virou RVTY em mai/2023
    "RE",    # Everest Re Group, saiu em jun/2023
    "TMK",   # Torchmark → GL em ago/2020 (limite, mas tentamos)
    "WRK",   # WestRock, fundiu com Smurfit em jun/2024
]

rel_val = pd.read_csv(CHECKPOINT_VAL)
ja_ok   = set(rel_val.loc[rel_val["status"] == "ok", "ticker"])
pendentes = [t for t in RETRY_YAHOO if t not in ja_ok]
print(f"Tentando Yahoo (Adj Close) para {len(pendentes)} tickers...")
print()

for ticker in pendentes:
    print(f"  {ticker}... ", end="", flush=True)
    try:
        raw = yf.download(ticker, start="2015-07-01", end="2026-01-01",
                          auto_adjust=True, progress=False)
        # auto_adjust=True → "Close" = Adj Close
        if isinstance(raw.columns, pd.MultiIndex):
            close = raw[("Close", ticker)].rename(ticker)
        else:
            close = raw["Close"].rename(ticker)
        close = close.dropna()

        if len(close) > 10:
            close.to_csv(PRICES_YAHOO / f"{ticker}.csv", header=True)
            row = {
                "ticker":      ticker,
                "status":      "ok",
                "n_rows":      len(close),
                "price_from":  close.index[0].strftime("%Y-%m-%d"),
                "price_until": close.index[-1].strftime("%Y-%m-%d"),
            }
            print(f"ok ({len(close)} linhas | {row['price_from']} → {row['price_until']})")
        else:
            row = {"ticker": ticker, "status": "sem dados yahoo",
                   "n_rows": len(close), "price_from": "—", "price_until": "—"}
            print(f"sem dados ({len(close)} linhas)")

    except Exception as e:
        row = {"ticker": ticker, "status": "erro",
               "n_rows": 0, "price_from": "—", "price_until": str(e)[:80]}
        print(f"erro: {e}")

    rel_val = rel_val[rel_val["ticker"] != ticker]
    rel_val = pd.concat([rel_val, pd.DataFrame([row])], ignore_index=True)
    rel_val.to_csv(CHECKPOINT_VAL, index=False)
    time.sleep(0.3)

print()
print("=== Resultado ===")
print(rel_val["status"].value_counts().to_string())
print()
print(rel_val.to_string(index=False))

Tentando Yahoo (Adj Close) para 12 tickers...

  ANTM... 

$ANTM: possibly delisted; no timezone found

1 Failed download:
['ANTM']: possibly delisted; no timezone found


sem dados (0 linhas)
  BLL... 

$BLL: possibly delisted; no timezone found

1 Failed download:
['BLL']: possibly delisted; no timezone found


sem dados (0 linhas)
  CDAY... 

...

sem dados (0 linhas)
  TMK... 

$TMK: possibly delisted; no timezone found

1 Failed download:
['TMK']: possibly delisted; no timezone found


sem dados (0 linhas)
  WRK... 

$WRK: possibly delisted; no timezone found

1 Failed download:
['WRK']: possibly delisted; no timezone found


sem dados (0 linhas)

=== Resultado ===
status
ok                  84
sem dados yahoo     12
sem dados            7
erro_http_404        6
parcial_sem_21CF     2
lacuna_legitima      1
parcial_sem_2016     1

ticker           status  n_rows price_from price_until
   FOX parcial_sem_21CF    1711 2019-03-13  2025-12-30
  FOXA parcial_sem_21CF    1712 2019-03-12  2025-12-30
    IR parcial_sem_2016    2171 2017-05-12  2025-12-30
   DOW  lacuna_legitima    1706 2019-03-20  2025-12-30
  AABA               ok    1076 2015-07-01  2019-11-06
  ABMD               ok    1886 2015-07-01  2023-01-03
   AGN               ok    1223 2015-07-01  2020-05-08
  ALXN               ok    1527 2015-07-01  2021-07-28
  ANSS               ok    2526 2015-07-01  2025-07-17
  ARNC               ok     851 2020-04-01  2023-08-17
  ATVI               ok    2087 2015-07-01  2023-10-13
   BCR               ok     631 2015-07-01  2017-12-29
  BHGE               ok    2136 2017-07-05  2025-12-31
  CELG               

In [54]:
print("=" * 65)
print("RESUMO DAS AÇÕES NECESSÁRIAS")
print("=" * 65)

# [A] Tickers do report com lacuna de dados na extensão
acao_a = [r for r in rows_cov if r["resultado"] not in ("ok", "prof. cobre")]
if acao_a:
    print("\n[A] Lacunas de dados nos tickers do report:")
    for r in acao_a:
        print(f"    {r['ticker']:8s}  saiu do índice em ~{r['sp500_until']:12s}  "
              f"dados até {r['price_until']:12s}  → {r['resultado']}")
else:
    print("\n[A] Sem lacunas nos tickers do report.")

# [B] Sucessores sem arquivo de preço
acao_b = [r for r in rows_suc if r["arquivo"] == "NÃO TEM"]
if acao_b:
    print("\n[B] Sucessores sem arquivo de preço:")
    for r in acao_b:
        print(f"    {r['antigo']:8s} → {r['novo']:6s}  entrou no sp500 em {r['novo_sp500']}")
else:
    print("\n[B] Todos os sucessores têm arquivo de preço.")

# [C] Cobertos pelo professor
ok_prof = [r for r in rows_cov if r["resultado"] == "prof. cobre"]
if ok_prof:
    print(f"\n[C] Cobertos pela base do professor (saíram do índice antes de 2016):")
    for r in ok_prof:
        print(f"    {r['ticker']:8s}  última aparição: {r['sp500_until']}")

RESUMO DAS AÇÕES NECESSÁRIAS

[A] Lacunas de dados nos tickers do report:
    ABC       saiu do índice em ~2023-08-25    dados até —             → SEM ARQUIVO
    ADS       saiu do índice em ~2020-05-22    dados até —             → SEM ARQUIVO
    ANTM      saiu do índice em ~2022-06-21    dados até —             → SEM ARQUIVO
    ANTM      saiu do índice em ~2022-06-21    dados até —             → SEM ARQUIVO
    APC       saiu do índice em ~2019-08-08    dados até —             → SEM ARQUIVO
    APC       saiu do índice em ~2019-08-08    dados até —             → SEM ARQUIVO
    BF.B      saiu do índice em ~2026-01-14    dados até —             → SEM ARQUIVO
    BLL       saiu do índice em ~2022-04-11    dados até —             → SEM ARQUIVO
    BLL       saiu do índice em ~2022-04-11    dados até —             → SEM ARQUIVO
    BRK.B     saiu do índice em ~2026-01-14    dados até —             → SEM ARQUIVO
    CBS       saiu do índice em ~2019-11-21    dados até —             → SEM

---
## Parte 9 — Limpar arquivos do grupo [B] com dados incorretos

Os tickers do grupo [B] têm arquivos CSV, mas os dados são de empresas **diferentes** que
reciclaram o ticker após a empresa original sair do S&P 500. Precisamos apagar esses arquivos
antes de tentar buscar os dados corretos no Tiingo.

In [55]:
rel_val = pd.read_csv(CHECKPOINT_VAL)

# Tickers [B] cujos arquivos Yahoo têm dados de outra empresa (ticker reciclado)
# Os arquivos FOX e FOXA NÃO são deletados — têm dados válidos da Fox Corp 2019+
# (queremos só pré-pender os dados 2016–2018 da 21st Century Fox via Tiingo)
WRONG_YAHOO = {
    "CA":   "CA Technologies adquirida Broadcom nov/2018; arquivo tem 2023+",
    "EMC":  "EMC adquirida Dell set/2016; arquivo tem 2023+",
    "FB":   "Facebook virou Meta out/2021; arquivo tem 2025+",
    "KORS": "Michael Kors virou CPRI dez/2017; arquivo tem jan–jun 2018",
    "LB":   "L Brands virou BBWI ago/2021; arquivo tem 2024+",
    "SE":   "Spectra Energy adquirida Enbridge fev/2017; arquivo tem 2017-10+",
    "STI":  "SunTrust fundiu com BB&T dez/2019; arquivo tem 2022+",
    "TE":   "TECO Energy adquirida Emera jul/2016; arquivo tem 2020+",
}
WRONG_TIINGO = {
    "VIAC": "arquivo Tiingo tem dados pré-fusão (jul–dez 2019); ViacomCBS era 2020-2022",
}

print("Deletando arquivos com dados incorretos (grupo [B]):")
for ticker, motivo in {**WRONG_YAHOO, **WRONG_TIINGO}.items():
    folder = PRICES_YAHOO if ticker in WRONG_YAHOO else PRICES_TIINGO
    f = folder / f"{ticker}.csv"
    if f.exists():
        f.unlink()
        print(f"  {ticker:6s}  deletado  ({motivo})")
    else:
        print(f"  {ticker:6s}  já não existe")

    # Atualiza checkpoint
    rel_val = rel_val[rel_val["ticker"] != ticker]
    rel_val = pd.concat([rel_val, pd.DataFrame([{
        "ticker": ticker, "status": "deletado_ticker_reciclado",
        "n_rows": 0, "price_from": "—", "price_until": "—"
    }])], ignore_index=True)

rel_val.to_csv(CHECKPOINT_VAL, index=False)
print("\nCheckpoint atualizado.")

Deletando arquivos com dados incorretos (grupo [B]):
  CA      já não existe
  EMC     já não existe
  FB      já não existe
  KORS    já não existe
  LB      já não existe
  SE      já não existe
  STI     já não existe
  TE      já não existe
  VIAC    deletado  (arquivo Tiingo tem dados pré-fusão (jul–dez 2019); ViacomCBS era 2020-2022)

Checkpoint atualizado.


---
## Parte 10 — Tiingo com datas específicas por ticker

Estratégia: consultar Tiingo passando `startDate` e `endDate` dentro do período real de
membership de cada empresa. Como o Tiingo mantém histórico de tickers delistados por data,
limitar o `endDate` à data de saída do índice deve retornar os dados da empresa **original**
(não da empresa que reciclou o ticker mais tarde).

Após baixar, validamos se o retorno cobre o período esperado.

**Casos especiais:**
- `FOX`, `FOXA`: arquivo existente tem Fox Corp (2019+) — queremos pré-pender dados da
  21st Century Fox (2016–2018) via Tiingo, depois concatenar com o arquivo atual.
- `IR`: arquivo existente começa em mai/2017 — queremos pré-pender 2016 via Tiingo.

In [56]:
import requests, time
import pandas as pd
from pathlib import Path

TIINGO_TOKEN = "dadfd331f2cb44969b8f7468006d20ad62b13262"
rel_val = pd.read_csv(CHECKPOINT_VAL)

# (startDate, endDate) = período de membership da empresa ORIGINAL no S&P 500
TARGETS = {
    # ── Grupo [A] — sem arquivo ───────────────────────────────────────────────
    "ANTM":  ("2015-07-01", "2022-06-28"),   # Anthem → Elevance Health (ELV)
    "APC":   ("2015-07-01", "2019-08-08"),   # Anadarko → OXY
    "BLL":   ("2015-07-01", "2022-04-11"),   # Ball Corporation
    "CDAY":  ("2021-09-20", "2023-10-18"),   # Ceridian HCM
    "FBHS":  ("2016-06-01", "2022-11-08"),   # Fortune Brands → FBIN
    "FRC":   ("2018-07-01", "2023-05-01"),   # First Republic Bank (falência)
    "GPS":   ("2015-07-01", "2022-01-10"),   # The Gap
    "MMC":   ("2015-07-01", "2025-12-31"),   # Marsh & McLennan (ainda ativo!)
    "PEAK":  ("2019-07-01", "2024-02-01"),   # Healthpeak Properties (→ DOC)
    "PKI":   ("2015-07-01", "2023-05-04"),   # PerkinElmer → Revvity
    "RE":    ("2015-07-01", "2023-06-20"),   # Everest Re Group
    "TMK":   ("2015-07-01", "2019-08-08"),   # Torchmark → Globe Life (GL)
    "WRK":   ("2015-07-01", "2024-07-05"),   # WestRock → Smurfit WestRock
    # ── Grupo [B] — arquivo apagado, busca dados corretos ────────────────────
    "CA":    ("2015-07-01", "2018-11-05"),   # CA Technologies → Broadcom
    "EMC":   ("2015-07-01", "2016-09-07"),   # EMC Corporation → Dell
    "FB":    ("2015-07-01", "2021-10-28"),   # Facebook → Meta Platforms
    "KORS":  ("2015-07-01", "2017-12-31"),   # Michael Kors → Capri Holdings
    "LB":    ("2015-07-01", "2021-08-02"),   # L Brands → BBWI
    "SE":    ("2015-07-01", "2017-02-27"),   # Spectra Energy → Enbridge
    "STI":   ("2015-07-01", "2019-12-06"),   # SunTrust Banks → Truist (TFC)
    "TE":    ("2015-07-01", "2016-07-01"),   # TECO Energy → Emera
    "VIAC":  ("2019-12-05", "2022-02-15"),   # ViacomCBS → Paramount (PARA)
    # ── Prepend: arquivo existente (Yahoo) cobre datas posteriores ────────────
    "FOX":   ("2015-07-01", "2019-03-18"),   # 21st Century Fox (não-votante)
    "FOXA":  ("2015-07-01", "2019-03-18"),   # 21st Century Fox (votante)
    "IR":    ("2015-07-01", "2017-05-11"),   # Ingersoll-Rand original (→ TT)
}

PREPEND = {"FOX", "FOXA", "IR"}

def find_existing(ticker):
    for folder in [PRICES_YAHOO, PRICES_TIINGO]:
        f = folder / f"{ticker}.csv"
        if f.exists():
            return f, folder
    return None, None

def fetch_tiingo_range(ticker, start, end):
    """Consulta Tiingo — retorna adjClose (Adj Close). Retorna (Series, status)."""
    r = requests.get(
        f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
        params={
            "startDate":    start,
            "endDate":      end,
            "token":        TIINGO_TOKEN,
            "resampleFreq": "daily",
        },
        timeout=25,
    )
    if r.status_code != 200:
        return None, f"HTTP {r.status_code}"
    data = r.json()
    if not data:
        return None, "sem dados"
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert(None).dt.normalize()
    df = df.set_index("date").sort_index()
    if "adjClose" not in df.columns or df["adjClose"].isna().all():
        return None, "sem coluna adjClose"
    return df["adjClose"].rename(ticker).dropna(), "ok"

print(f"Buscando {len(TARGETS)} tickers no Tiingo (adjClose) com datas específicas...\n")
resultados = []

for ticker, (start, end) in TARGETS.items():
    print(f"  {ticker:6s} ({start} → {end})... ", end="", flush=True)

    serie, status = fetch_tiingo_range(ticker, start, end)

    if serie is None or len(serie) < 5:
        print(f"FALHOU — {status}")
        resultados.append({"ticker": ticker, "status": f"tiingo_{status}",
                            "n_rows": 0, "price_from": "—", "price_until": "—",
                            "primeiro_preco": "—"})
        time.sleep(0.3)
        continue

    got_start = serie.index.min()
    got_end   = serie.index.max()
    end_dt    = pd.Timestamp(end)

    if got_start > end_dt:
        print(f"DADOS ERRADOS — começa em {got_start.date()}, esperado antes de {end}")
        resultados.append({"ticker": ticker, "status": "tiingo_ticker_reciclado",
                            "n_rows": len(serie),
                            "price_from":  got_start.strftime("%Y-%m-%d"),
                            "price_until": got_end.strftime("%Y-%m-%d"),
                            "primeiro_preco": f"{serie.iloc[0]:.2f}"})
        time.sleep(0.3)
        continue

    if ticker in PREPEND:
        existing_path, existing_folder = find_existing(ticker)
        if existing_path is not None:
            existing = pd.read_csv(existing_path, index_col=0, parse_dates=True)
            existing.index = pd.to_datetime(existing.index, utc=True).tz_convert(None).normalize()
            existing_serie = existing.iloc[:, 0].rename(ticker)
            existing_filtrado = existing_serie[existing_serie.index > end_dt]
            combined = pd.concat([serie, existing_filtrado]).sort_index()
            combined = combined[~combined.index.duplicated(keep="last")]
            combined.to_csv(existing_path, header=True)
            n_salvo   = len(combined)
            save_info = f"prepend em {existing_folder.name}"
            disp_start = combined.index.min().date()
            disp_end   = combined.index.max().date()
        else:
            serie.to_csv(PRICES_TIINGO / f"{ticker}.csv", header=True)
            n_salvo   = len(serie)
            save_info = "novo em prices_tiingo"
            disp_start = got_start.date()
            disp_end   = got_end.date()
    else:
        serie.to_csv(PRICES_TIINGO / f"{ticker}.csv", header=True)
        n_salvo   = len(serie)
        save_info = "novo em prices_tiingo"
        disp_start = got_start.date()
        disp_end   = got_end.date()

    print(f"ok — {n_salvo} linhas ({disp_start} → {disp_end})  P0={serie.iloc[0]:.2f}  [{save_info}]")

    resultados.append({
        "ticker":         ticker,
        "status":         "ok_tiingo",
        "n_rows":         n_salvo,
        "price_from":     str(disp_start),
        "price_until":    str(disp_end),
        "primeiro_preco": f"{serie.iloc[0]:.2f}",
    })

    row_chk = {"ticker": ticker, "status": "ok", "n_rows": n_salvo,
               "price_from": str(disp_start), "price_until": str(disp_end)}
    rel_val = rel_val[rel_val["ticker"] != ticker]
    rel_val = pd.concat([rel_val, pd.DataFrame([row_chk])], ignore_index=True)
    rel_val.to_csv(CHECKPOINT_VAL, index=False)
    time.sleep(0.4)

print()
df_res = pd.DataFrame(resultados)
ok    = df_res[df_res["status"] == "ok_tiingo"]
falha = df_res[df_res["status"] != "ok_tiingo"]

print(f"{'='*65}")
print(f"RESULTADO  ok={len(ok)}  falha={len(falha)}")
print(f"{'='*65}")
if len(ok):
    print("\nRecuperados com sucesso:")
    print(ok[["ticker","n_rows","price_from","price_until","primeiro_preco"]].to_string(index=False))
if len(falha):
    print("\nNão recuperados:")
    print(falha[["ticker","status","price_from"]].to_string(index=False))

Buscando 25 tickers no Tiingo (adjClose) com datas específicas...

  ANTM   (2015-07-01 → 2022-06-28)... FALHOU — sem dados
  APC    (2015-07-01 → 2019-08-08)... FALHOU — sem dados
  BLL    (2015-07-01 → 2022-04-11)... FALHOU — sem dados
  CDAY   (2021-09-20 → 2023-10-18)... FALHOU — HTTP 404
  FBHS   (2016-06-01 → 2022-11-08)... FALHOU — sem dados
  FRC    (2018-07-01 → 2023-05-01)... FALHOU — HTTP 404
  GPS    (2015-07-01 → 2022-01-10)... FALHOU — sem dados
  MMC    (2015-07-01 → 2025-12-31)... FALHOU — sem dados
  PEAK   (2019-07-01 → 2024-02-01)... FALHOU — sem dados
  PKI    (2015-07-01 → 2023-05-04)... FALHOU — sem dados
  RE     (2015-07-01 → 2023-06-20)... FALHOU — sem dados
  TMK    (2015-07-01 → 2019-08-08)... FALHOU — sem dados
  WRK    (2015-07-01 → 2024-07-05)... FALHOU — HTTP 429
  CA     (2015-07-01 → 2018-11-05)... FALHOU — HTTP 429
  EMC    (2015-07-01 → 2016-09-07)... FALHOU — HTTP 429
  FB     (2015-07-01 → 2021-10-28)... FALHOU — HTTP 429
  KORS   (2015-07-01 → 2017